In [2]:
import os
import nibabel as nib
import numpy as np

# Tu ruta de red compartida
base_network_dir = r"\\smb.iib.uam.es\mdnovalbos\info_services\blizarbe_lab_share\TFG Lola\NEXI\Diffusion_times"

atlas_file = os.path.join(base_network_dir, "MouseX-Allen-Atlas/MouseX/MouseX-DW-ALLEN_Template_T2W.nii")
corrected_atlas_file = os.path.join(base_network_dir, "MouseX-Allen-Atlas/MouseX/MouseX-DW-ALLEN_Template_T2W_corrected.nii")

print("--- CORRECTING TEMPLATE ---")
nii_atlas = nib.load(atlas_file)
data_atlas = nii_atlas.get_fdata()
affine_atlas = nii_atlas.affine.copy()
header_atlas = nii_atlas.header.copy()

# Correct FOV (100mm -> 10mm)
scale_factor = 0.1
affine_atlas[:3, :3] *= scale_factor
header_atlas.set_zooms(tuple(z * scale_factor for z in header_atlas.get_zooms()))

# Invert vertically
data_flipped_atlas = np.flip(data_atlas, axis=1)
ny = data_atlas.shape[1]
affine_atlas[:3, 1] *= -1
affine_atlas[:3, 3] += affine_atlas[:3, 1] * (ny - 1)

nii_corrected_atlas = nib.Nifti1Image(data_flipped_atlas, affine_atlas, header_atlas)
nib.save(nii_corrected_atlas, corrected_atlas_file)

print("--- CORRECTING ANNOTATION LABELS ---")
annotation_file = os.path.join(base_network_dir, "MouseX-Allen-Atlas/MouseX/MouseX-DW-ALLEN_Annotation.nii")
corrected_annotation_file = os.path.join(base_network_dir, "MouseX-Allen-Atlas/MouseX/MouseX-DW-ALLEN_Annotation_corrected.nii")

nii_annot = nib.load(annotation_file)
data_annot = nii_annot.get_fdata()
affine_annot = nii_annot.affine.copy()
header_annot = nii_annot.header.copy()

affine_annot[:3, :3] *= scale_factor
header_annot.set_zooms(tuple(z * scale_factor for z in header_annot.get_zooms()))
data_flipped_annot = np.flip(data_annot, axis=1)
affine_annot[:3, 1] *= -1
affine_annot[:3, 3] += affine_annot[:3, 1] * (ny - 1)

nii_corrected_annot = nib.Nifti1Image(data_flipped_annot, affine_annot, header_annot)
nib.save(nii_corrected_annot, corrected_annotation_file)
print("Atlas preparation completed.")

--- CORRECTING TEMPLATE ---
--- CORRECTING ANNOTATION LABELS ---
Atlas preparation completed.


In [20]:
!C:\Users\Estudiantes\anaconda3\python.exe -m pip uninstall -y scipy numpy antspyx
!C:\Users\Estudiantes\anaconda3\python.exe -m pip install --only-binary :all: numpy==2.1.3 scipy==1.14.1 antspyx==0.6.3

Found existing installation: scipy 1.15.3
Uninstalling scipy-1.15.3:
  Successfully uninstalled scipy-1.15.3
Found existing installation: numpy 2.3.5
Uninstalling numpy-2.3.5:
  Successfully uninstalled numpy-2.3.5
Found existing installation: antspyx 0.6.3
Uninstalling antspyx-0.6.3:
  Successfully uninstalled antspyx-0.6.3


You can safely remove it manually.
You can safely remove it manually.


  Using cached numpy-2.1.3-cp313-cp313-win_amd64.whl.metadata (60 kB)
  Using cached scipy-1.14.1-cp313-cp313-win_amd64.whl.metadata (60 kB)
  Using cached antspyx-0.6.3-cp313-cp313-win_amd64.whl.metadata (7.2 kB)
   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
   ---------------------------------------  12.3/12.6 MB 99.3 MB/s eta 0:00:01
   ---------------------------------------- 12.6/12.6 MB 67.5 MB/s  0:00:00
   ---------------------------------------- 0.0/44.5 MB ? eta -:--:--
   -- ------------------------------------- 3.1/44.5 MB 14.8 MB/s eta 0:00:03
   --- ------------------------------------ 4.2/44.5 MB 13.2 MB/s eta 0:00:04
   ------ --------------------------------- 7.3/44.5 MB 11.0 MB/s eta 0:00:04
   --------- ------------------------------ 10.5/44.5 MB 11.9 MB/s eta 0:00:03
   ----------- ---------------------------- 12.6/44.5 MB 12.5 MB/s eta 0:00:03
   -------------- -----------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
resomapper 1.0.0 requires dipy==1.8.0, but you have dipy 1.11.0 which is incompatible.
resomapper 1.0.0 requires matplotlib<4.0.0,>=3.10.7, but you have matplotlib 3.10.6 which is incompatible.
resomapper 1.0.0 requires nibabel==5.3.2, but you have nibabel 5.3.3 which is incompatible.
resomapper 1.0.0 requires numpy==1.26.4, but you have numpy 2.1.3 which is incompatible.
resomapper 1.0.0 requires opencv-python==4.11.0.86, but you have opencv-python 4.13.0.92 which is incompatible.
resomapper 1.0.0 requires pydicom==3.0.1, but you have pydicom 3.0.2 which is incompatible.
resomapper 1.0.0 requires scikit-learn==1.6.1, but you have scikit-learn 1.7.2 which is incompatible.
resomapper 1.0.0 requires simpleitk==2.4.1, but you have simpleitk 2.5.3 which is incompatible.


In [1]:
import os
import nibabel as nib
import numpy as np
import ants

base_network_dir = r"\\smb.iib.uam.es\mdnovalbos\info_services\blizarbe_lab_share\TFG Lola\NEXI\Diffusion_times"
template_output_path = os.path.join(base_network_dir, "Study_T2w_Template.nii.gz")

# Códigos de tus 8 ratones
mis_ratones = ["R85", "R86", "R87", "R88", "R98", "R99", "R105", "R107"]
number_slices = 5 

print("--- BUSCANDO TUS CEREBROS RE CORTADOS ---")
population_files = []

for raton in mis_ratones:
    # Construimos la ruta exacta BIDS que se ve en tu captura de pantalla
    # Buscamos la carpeta 'anat' dentro de la subcarpeta que empieza por 'sub-'
    path_raton_raiz = os.path.join(base_network_dir, raton, "sourcedata")
    
    if os.path.exists(path_raton_raiz):
        subcarpetas = [f for f in os.listdir(path_raton_raiz) if f.startswith("sub-")]
        if subcarpetas:
            archivo_masked = os.path.join(path_raton_raiz, subcarpetas[0], "anat", f"{subcarpetas[0]}_acq-6_run-1_T2w_masked.nii")
            if os.path.exists(archivo_masked):
                population_files.append(archivo_masked)
                print(f"✓ Encontrado cerebro limpio para: {raton}")
            else:
                # Intento alternativo por si cambia el acq-X o run-X en algún ratón
                import glob
                lista_alt = glob.glob(os.path.join(path_raton_raiz, subcarpetas[0], "anat", "*_T2w_masked.nii*"))
                if lista_alt:
                    population_files.append(lista_alt[0])
                    print(f"✓ Encontrado (alternativo) para: {raton}")

if len(population_files) < len(mis_ratones):
    print(f"⚠️ Alerta: Solo encontré {len(population_files)} de los 8 ratones. Revisa los nombres.")

print("\n--- GENERANDO TEMPLATE POR CORTE (CÓDIGO ORIGINAL) ---")
for slice_id in range(number_slices):
    print(f"Procesando rodaja {slice_id}...")
    population_slice = [ants.image_read(f)[:, :, slice_id] for f in population_files]
    
    template_slice = ants.build_template(initial_template=None, image_list=population_slice,
                                         syn_sampling=2, reg_iterations=(100, 70, 50, 40),
                                         grad_step=0.05, syn_metric='CC', flow_sigma=4, total_sigma=1,
                                         type_of_transformation='SyN')
    template_slice.image_write(template_output_path.replace(".nii.gz", f"_{slice_id}.nii.gz"))

nii_images = [nib.load(template_output_path.replace(".nii.gz", f"_{i}.nii.gz")) for i in range(number_slices)]
stack = np.stack([img.get_fdata() for img in nii_images], axis=-1)
ref_img = nib.load(population_files[0])
new_nifti = nib.Nifti1Image(stack, ref_img.affine, ref_img.header)
nib.save(new_nifti, template_output_path)
print(f"--- TEMPLATE COMPLETADO: {template_output_path} ---")

--- BUSCANDO TUS CEREBROS RE CORTADOS ---
✓ Encontrado (alternativo) para: R85
✓ Encontrado (alternativo) para: R86
✓ Encontrado (alternativo) para: R87
✓ Encontrado (alternativo) para: R88
✓ Encontrado (alternativo) para: R98
✓ Encontrado (alternativo) para: R99
✓ Encontrado (alternativo) para: R105
✓ Encontrado (alternativo) para: R107

--- GENERANDO TEMPLATE POR CORTE (CÓDIGO ORIGINAL) ---
Procesando rodaja 0...
['2', 'C:\\Users\\ESTUDI~1\\AppData\\Local\\Temp\\tmpa931d0ry.mat', 'C:\\Users\\ESTUDI~1\\AppData\\Local\\Temp\\tmp6vxv1b0t\\img0000\\out0GenericAffine.mat', 'C:\\Users\\ESTUDI~1\\AppData\\Local\\Temp\\tmp6vxv1b0t\\img0001\\out0GenericAffine.mat', 'C:\\Users\\ESTUDI~1\\AppData\\Local\\Temp\\tmp6vxv1b0t\\img0002\\out0GenericAffine.mat', 'C:\\Users\\ESTUDI~1\\AppData\\Local\\Temp\\tmp6vxv1b0t\\img0003\\out0GenericAffine.mat', 'C:\\Users\\ESTUDI~1\\AppData\\Local\\Temp\\tmp6vxv1b0t\\img0004\\out0GenericAffine.mat', 'C:\\Users\\ESTUDI~1\\AppData\\Local\\Temp\\tmp6vxv1b0t\\img000

In [3]:
import os
import ants
import nibabel as nib
import numpy as np

# Usamos la barra invertida de red exacta que tienes en tu máquina
base_network_dir = r"\\smb.iib.uam.es\mdnovalbos\info_services\blizarbe_lab_share\TFG Lola\NEXI\Diffusion_times"

# Nombres de archivos 100% idénticos a los que generó tu Celda 0 original
atlas_template_path = os.path.join(base_network_dir, "MouseX-Allen-Atlas", "MouseX", "MouseX-DW-ALLEN_Template_T2W_corrected.nii")
atlas_labels_path = os.path.join(base_network_dir, "MouseX-Allen-Atlas", "MouseX", "MouseX-DW-ALLEN_Annotation_corrected.nii")
study_template_path = os.path.join(base_network_dir, "Study_T2w_Template.nii.gz")

# Tu correspondencia exacta para tus 5 cortes reales
slice_correspondence = [(0, 11), (1, 14), (2, 17), (3, 20), (4, 23)]

# Nombres de salida originales
output_atlas_path = os.path.join(base_network_dir, "Study_MouseX_warpedAtlas.nii.gz")
output_labels_path = os.path.join(base_network_dir, "Study_Template_Annotation.nii.gz")

print("--- INICIANDO REGISTRO DE ANTS (ATLAS -> TEMPLATE) ---")
atlas_3d = ants.image_read(atlas_template_path)
atlas_labels_3d = ants.image_read(atlas_labels_path)
study_template_3d = ants.image_read(study_template_path)

atlas_warped_slices = []
atlas_labels_warped_slices = []

for i, j in slice_correspondence:
    study_template_slice = study_template_3d[:, :, i]
    atlas_slice = atlas_3d[:, :, j]
    atlas_labels_slice = atlas_labels_3d[:, :, j]

    init = ants.affine_initializer(study_template_slice, atlas_slice)
    reg = ants.registration(fixed=study_template_slice, moving=atlas_slice,
                            syn_sampling=2, reg_iterations=(100, 70, 50, 40),
                            grad_step=0.05, initial_transform=init, syn_metric='CC',
                            flow_sigma=4, total_sigma=1, type_of_transformation='SyN')

    atlas_labels_warped = ants.apply_transforms(fixed=study_template_slice, moving=atlas_labels_slice,
                                                transformlist=reg["fwdtransforms"], interpolator='genericLabel')

    atlas_warped_slices.append(reg["warpedmovout"].numpy())
    atlas_labels_warped_slices.append(atlas_labels_warped.numpy())

atlas_warped_3d = np.stack(atlas_warped_slices, axis=2)
atlas_labels_warped_3d = np.stack(atlas_labels_warped_slices, axis=2)
tmpl_nib = nib.load(study_template_path)

nib.save(nib.Nifti1Image(atlas_warped_3d, tmpl_nib.affine, tmpl_nib.header), output_atlas_path)
nib.save(nib.Nifti1Image(atlas_labels_warped_3d, tmpl_nib.affine, tmpl_nib.header), output_labels_path)
print("--- ¡EL ATLAS SE HA ADAPTADO A TU TEMPLATE DE 5 CORTES CON ÉXITO! ---")

--- INICIANDO REGISTRO DE ANTS (ATLAS -> TEMPLATE) ---
--- ¡EL ATLAS SE HA ADAPTADO A TU TEMPLATE DE 5 CORTES CON ÉXITO! ---


In [6]:
import os
import ants
import nibabel as nib
import numpy as np

base_network_dir = r"\\smb.iib.uam.es\mdnovalbos\info_services\blizarbe_lab_share\TFG Lola\NEXI\Diffusion_times"

template_path = os.path.join(base_network_dir, "Study_T2w_Template.nii.gz")
labels_path = os.path.join(base_network_dir, "Study_Template_Annotation.nii.gz")
mis_ratones = ["R85", "R86", "R87", "R88", "R98", "R99", "R105", "R107"]

template_3d = ants.image_read(template_path)
labels_3d = ants.image_read(labels_path)

print("--- LLEVANDO ETIQUETAS A CADA RATÓN ---")
for raton in mis_ratones:
    path_deriv = os.path.join(base_network_dir, raton, "derivatives")
    os.makedirs(path_deriv, exist_ok=True)
    
    # 1. Buscamos tu cerebro recortado real en 'sourcedata' (Ruta BIDS de tu captura)
    path_raton_raiz = os.path.join(base_network_dir, raton, "sourcedata")
    if not os.path.exists(path_raton_raiz):
        print(f"⚠️ No encuentro la carpeta sourcedata para el ratón {raton}")
        continue
        
    subcarpetas = [f for f in os.listdir(path_raton_raiz) if f.startswith("sub-")]
    if not subcarpetas:
        print(f"⚠️ No encuentro subcarpeta sub- dentro de sourcedata para {raton}")
        continue
        
    subject_t2w_file = os.path.join(path_raton_raiz, subcarpetas[0], "anat", f"{subcarpetas[0]}_acq-6_run-1_T2w_masked.nii")
    
    # Verificación por si algún ratón no tiene exactamente el 'acq-6_run-1'
    if not os.path.exists(subject_t2w_file):
        import glob
        lista_alt = glob.glob(os.path.join(path_raton_raiz, subcarpetas[0], "anat", "*_T2w_masked.nii*"))
        if lista_alt:
            subject_t2w_file = lista_alt[0]
        else:
            print(f"⚠️ No encuentro el archivo _T2w_masked.nii para {raton}")
            continue

    # 2. Si el archivo existe, cargamos y registramos corte a corte
    subject_t2w_3d = ants.image_read(subject_t2w_file)
    atlas_labels_warped_slices = []
    
    for i in range(5):  # Tus 5 cortes reales
        template_slice = template_3d[:, :, i]
        labels_slice = labels_3d[:, :, i]
        subject_t2w_slice = subject_t2w_3d[:, :, i]

        init = ants.affine_initializer(subject_t2w_slice, template_slice)
        reg = ants.registration(fixed=subject_t2w_slice, moving=template_slice,
                                syn_sampling=2, reg_iterations=(100, 70, 50, 40),
                                grad_step=0.05, initial_transform=init, syn_metric='CC',
                                flow_sigma=4, total_sigma=1, type_of_transformation='SyN')

        atlas_labels_warped = ants.apply_transforms(fixed=subject_t2w_slice, moving=labels_slice,
                                                    transformlist=reg["fwdtransforms"], interpolator='genericLabel')
        atlas_labels_warped_slices.append(atlas_labels_warped.numpy())

    atlas_labels_warped_3d = np.stack(atlas_labels_warped_slices, axis=2)
    tmpl_nib = nib.load(template_path)
    
    # 3. Guardamos el resultado final en la carpeta derivatives de ese ratón
    output_labels_path = os.path.join(path_deriv, f"{raton}_individual_labels.nii.gz")
    nib.save(nib.Nifti1Image(atlas_labels_warped_3d, tmpl_nib.affine, tmpl_nib.header), output_labels_path)
    print(f"✓ Etiquetas individuales listas para ratón: {raton}")

--- LLEVANDO ETIQUETAS A CADA RATÓN ---
✓ Etiquetas individuales listas para ratón: R85
✓ Etiquetas individuales listas para ratón: R86
✓ Etiquetas individuales listas para ratón: R87
✓ Etiquetas individuales listas para ratón: R88
✓ Etiquetas individuales listas para ratón: R98
✓ Etiquetas individuales listas para ratón: R99
✓ Etiquetas individuales listas para ratón: R105
✓ Etiquetas individuales listas para ratón: R107


In [7]:
import os
import ants

base_network_dir = r"\\smb.iib.uam.es\mdnovalbos\info_services\blizarbe_lab_share\TFG Lola\NEXI\Diffusion_times"
mis_ratones = ["R85", "R86", "R87", "R88", "R98", "R99", "R105", "R107"]

print("--- RESAMPLING TO SELECTION MATRIX (128x128x5) ---")
for raton in mis_ratones:
    path_deriv = os.path.join(base_network_dir, raton, "derivatives")
    individual_labels_path = os.path.join(path_deriv, f"{raton}_individual_labels.nii.gz")
    
    if not os.path.exists(individual_labels_path):
        print(f"⚠️ Alerta: No encontré las etiquetas individuales para {raton}")
        continue
        
    labels_subject_3d = ants.image_read(individual_labels_path)
    
    # Redimensionado exacto para tus mapas paramétricos de obesidad
    output_size = [128, 128, 5]  
    labels_subject_scaled = labels_subject_3d.resample_image(output_size, use_voxels=True, interp_type=1)
    
    final_scaled_path = os.path.join(path_deriv, f"{raton}_labels_scaled.nii.gz")
    labels_subject_scaled.image_write(final_scaled_path)
    print(f"✓ Resuestreo a 128x128 completado para {raton}.")

print("--- PIPELINE FINALIZADO TOTALMENTE ---")

--- RESAMPLING TO SELECTION MATRIX (128x128x5) ---
✓ Resuestreo a 128x128 completado para R85.
✓ Resuestreo a 128x128 completado para R86.
✓ Resuestreo a 128x128 completado para R87.
✓ Resuestreo a 128x128 completado para R88.
✓ Resuestreo a 128x128 completado para R98.
✓ Resuestreo a 128x128 completado para R99.
✓ Resuestreo a 128x128 completado para R105.
✓ Resuestreo a 128x128 completado para R107.
--- PIPELINE FINALIZADO TOTALMENTE ---


In [9]:
import os
import glob
import nibabel as nib
import numpy as np
import pandas as pd

base_network_dir = r"\\smb.iib.uam.es\mdnovalbos\info_services\blizarbe_lab_share\TFG Lola\NEXI\Diffusion_times"
mis_ratones = ["R85", "R86", "R87", "R88", "R98", "R99", "R105", "R107"]

# Información de tus sujetos (Macho=1, Hembra=2 | LFLS=1, HFHS=2)
info_sujetos = {
    "R85":  {"Sex": 1, "Diet": 2}, "R86":  {"Sex": 1, "Diet": 2},
    "R87":  {"Sex": 2, "Diet": 1}, "R88":  {"Sex": 2, "Diet": 1},
    "R98":  {"Sex": 1, "Diet": 1}, "R99":  {"Sex": 1, "Diet": 1},
    "R105": {"Sex": 2, "Diet": 2}, "R107": {"Sex": 2, "Diet": 2}
}

tiempos_difusion = ["DKI_1_1", "DKI_2_1", "DKI_3_1", "DKI_4_1"]
metricas = ["MD", "AD", "RD", "AK", "MK", "RK", "FA_DKI"]
lista_areas = [1, 2, 3, 4, 5, 6, 7, 8]

filas_csv = []

print("--- INICIANDO EXTRACCIÓN DIRECTA ---")

for raton in mis_ratones:
    path_deriv = os.path.join(base_network_dir, raton, "derivatives")
    labels_path = os.path.join(path_deriv, f"{raton}_labels_scaled.nii.gz")
    
    if not os.path.exists(labels_path):
        print(f"⚠️ Saltando {raton}: No tiene el mapa de etiquetas.")
        continue
        
    mask_data = nib.load(labels_path).get_fdata()
    
    for tiempo in tiempos_difusion:
        mapas_cargados = {}
        for metrica in metricas:
            # RUTA FIJA DIRECTA: Buscamos directamente en dwi para ir rápido y evitar bucles infinitos en la red
            patron_directo = os.path.join(base_network_dir, raton, "sourcedata", "sub-*", "dwi", f"*{tiempo}*{metrica}*.nii*")
            archivos = glob.glob(patron_directo)
            
            if not archivos:
                # Segundo intento: búsqueda rápida solo en la carpeta de ese ratón
                patron_directo = os.path.join(base_network_dir, raton, "*", f"*{tiempo}*{metrica}*.nii*")
                archivos = glob.glob(patron_directo)
                
            if archivos:
                mapas_cargados[metrica] = nib.load(archivos[0]).get_fdata()
            else:
                mapas_cargados[metrica] = None

        for area_id in lista_areas:
            mascara_area = (mask_data == area_id)
            if np.any(mascara_area):
                data_fila = {"Mouse": raton, "Sex": info_sujetos[raton]["Sex"], "Diet": info_sujetos[raton]["Diet"],
                             "DiffusionTime": tiempo, "Area": area_id}
                
                for metrica in metricas:
                    if mapas_cargados[metrica] is not None:
                        data_fila[metrica] = np.mean(mapas_cargados[metrica][mascara_area])
                    else:
                        data_fila[metrica] = np.nan
                        
                filas_csv.append(data_fila)
                
    print(f"✓ Áreas extraídas para: {raton}")

# --- CAMBIO CLAVE: Guardamos en el Escritorio local del usuario 'Estudiantes' ---
df_final = pd.DataFrame(filas_csv)
desktop_path = r"C:\Users\Estudiantes\Desktop\MD_OUT_NuevasRegiones_Completo.csv"
df_final.to_csv(desktop_path, index=False)

print(f"\n🚀 ¡PROCESO COMPLETADO! Archivo guardado con permisos en tu Escritorio local:\n--> {desktop_path}")

--- INICIANDO EXTRACCIÓN DIRECTA ---
✓ Áreas extraídas para: R85
✓ Áreas extraídas para: R86
✓ Áreas extraídas para: R87
✓ Áreas extraídas para: R88
✓ Áreas extraídas para: R98
✓ Áreas extraídas para: R99
✓ Áreas extraídas para: R105
✓ Áreas extraídas para: R107

🚀 ¡PROCESO COMPLETADO! Archivo guardado con permisos en tu Escritorio local:
--> C:\Users\Estudiantes\Desktop\MD_OUT_NuevasRegiones_Completo.csv


In [12]:
import os
import glob

base_network_dir = r"\\smb.iib.uam.es\mdnovalbos\info_services\blizarbe_lab_share\TFG Lola\NEXI\Diffusion_times"

# Buscamos absolutamente todas las imágenes .nii dentro del R85, estén donde estén
patron_total = os.path.join(base_network_dir, "R86", "**", "*.nii*")
todos_los_nii = glob.glob(patron_total, recursive=True)

print("--- LOCALIZADOR DE MAPAS DE DIFUSIÓN ---")
if todos_los_nii:
    print(f"He encontrado {len(todos_los_nii)} archivos de imagen en el ratón R85.")
    print("\nAquí tienes las rutas reales de los primeros archivos para ver dónde se esconden:")
    # Te pinto los 8 primeros para ver la carpeta exacta
    for f in todos_los_nii[:8]:
        # Cortamos la ruta larga para que veas solo las carpetas internas
        ruta_corta = f.replace(base_network_dir, "")
        print(f"- {ruta_corta}")
else:
    print("❌ No encuentro ningún archivo .nii en R85. Mira en tu explorador de archivos de Windows cómo se llama la carpeta donde guardáis los mapas paramétricos (MD, AD, FA...).")

--- LOCALIZADOR DE MAPAS DE DIFUSIÓN ---
He encontrado 202 archivos de imagen en el ratón R85.

Aquí tienes las rutas reales de los primeros archivos para ver dónde se esconden:
- \R86\convertidos\convertido_20240923_164610_H230924_R86_tiemposdifusion_2_2309254_1_1\convertido_20240923_164610_H230924_R86_tiemposdifusion_2_2309254_1_1_11\convertido_20240923_164610_H230924_R86_tiemposdifusion_2_2309254_1_1_11.nii.gz
- \R86\convertidos\convertido_20240923_164610_H230924_R86_tiemposdifusion_2_2309254_1_1\convertido_20240923_164610_H230924_R86_tiemposdifusion_2_2309254_1_1_14\convertido_20240923_164610_H230924_R86_tiemposdifusion_2_2309254_1_1_14.nii.gz
- \R86\convertidos\convertido_20240923_164610_H230924_R86_tiemposdifusion_2_2309254_1_1\convertido_20240923_164610_H230924_R86_tiemposdifusion_2_2309254_1_1_18\convertido_20240923_164610_H230924_R86_tiemposdifusion_2_2309254_1_1_18.nii.gz
- \R86\convertidos\convertido_20240923_164610_H230924_R86_tiemposdifusion_2_2309254_1_1\convertido_202409

In [18]:
import os
import nibabel as nib
import numpy as np

base_network_dir = r"\\smb.iib.uam.es\mdnovalbos\info_services\blizarbe_lab_share\TFG Lola\NEXI\Diffusion_times"
labels_path = os.path.join(base_network_dir, "R85", "derivatives", "R85_labels_scaled.nii.gz")

if os.path.exists(labels_path):
    mask_data = nib.load(labels_path).get_fdata()
    # Buscamos qué números únicos e diferentes de cero hay pintados en la máscara
    numeros_reales = np.unique(mask_data)
    
    print("--- CONFESIÓN DE LA MÁSCARA ---")
    print(f"La máscara del R85 tiene {len(numeros_reales)} etiquetas diferentes.")
    print("\nLos primeros 15 números que he encontrado dentro de tu imagen son:")
    print(numeros_reales[:15])
else:
    print("❌ Increíble, ahora ni siquiera lee el archivo de etiquetas de R85.")

--- CONFESIÓN DE LA MÁSCARA ---
La máscara del R85 tiene 60 etiquetas diferentes.

Los primeros 15 números que he encontrado dentro de tu imagen son:
[ 0.          1.99996948  2.99995422  3.99993896  4.99992371  5.99990845
  6.99989319  7.99987793 10.99983215 11.99981689 12.99980164 13.99978638
 14.99977112 15.99975586 16.9997406 ]


In [20]:
import os
import glob
import nibabel as nib
import numpy as np
import pandas as pd

base_network_dir = r"\\smb.iib.uam.es\mdnovalbos\info_services\blizarbe_lab_share\TFG Lola\NEXI\Diffusion_times"
mis_ratones = ["R85", "R86", "R87", "R88", "R98", "R99", "R105", "R107"]

# Información de tus sujetos (Macho=1, Hembra=2 | LFLS=1, HFHS=2)
info_sujetos = {
    "R85":  {"Sex": 1, "Diet": 2}, "R86":  {"Sex": 1, "Diet": 2},
    "R87":  {"Sex": 2, "Diet": 1}, "R88":  {"Sex": 2, "Diet": 1},
    "R98":  {"Sex": 1, "Diet": 1}, "R99":  {"Sex": 1, "Diet": 1},
    "R105": {"Sex": 2, "Diet": 2}, "R107": {"Sex": 2, "Diet": 2}
}

tiempos_difusion = ["*difusion_1*", "*difusion_2*", "*difusion_3*", "*difusion_4*"]
tiempos_etiquetas = ["DKI_1_1", "DKI_2_1", "DKI_3_1", "DKI_4_1"]

# Ampliamos la lista para capturar todas las regiones que tiene tu atlas real
lista_areas = list(range(1, 65))

filas_csv = []

print("--- INICIANDO EXTRACCIÓN CON CORRECCIÓN DE REDONDEO ---")

for raton in mis_ratones:
    path_deriv = os.path.join(base_network_dir, raton, "derivatives")
    labels_path = os.path.normpath(os.path.join(path_deriv, f"{raton}_labels_scaled.nii.gz"))
    
    if not os.path.exists(labels_path):
        continue
        
    # Cargamos la máscara y... ¡PASO CLAVE!: Redondeamos los decimales a enteros limpios
    mask_data_raw = nib.load(labels_path).get_fdata()
    mask_data = np.round(mask_data_raw).astype(int)
    
    path_convertivos = os.path.normpath(os.path.join(base_network_dir, raton, "convertidos"))
    if not os.path.exists(path_convertivos):
        continue

    for t_idx, tiempo_patron in enumerate(tiempos_difusion):
        tiempo_r = tiempos_etiquetas[t_idx]
        
        patron_carpeta = os.path.join(path_convertivos, tiempo_patron)
        carpetas_tiempo = glob.glob(patron_carpeta)
        
        if not carpetas_tiempo:
            continue
            
        ruta_carpeta_efectiva = os.path.normpath(carpetas_tiempo[0])
        
        patron_total = os.path.join(ruta_carpeta_efectiva, "**", "*.nii*")
        todos_los_nii = [os.path.normpath(f) for f in glob.glob(patron_total, recursive=True)]
        
        mapas_filtrados = []
        for f in todos_los_nii:
            nombre = os.path.basename(f)
            if "_subvol_" not in nombre:
                mapas_filtrados.append(f)
                
        def extraer_id_numerico(filepath):
            nombre = os.path.basename(filepath).replace(".nii", "").replace(".gz", "")
            partes = nombre.split('_')
            for p in reversed(partes):
                if p.isdigit():
                    return int(p)
            return 0
            
        mapas_filtrados.sort(key=extraer_id_numerico)
        mapas_unicos = list(dict.fromkeys(mapas_filtrados))

        matrices_cargadas = []
        for f in mapas_unicos[:6]:
            try:
                matrices_cargadas.append(nib.load(f).get_fdata())
            except:
                matrices_cargadas.append(None)

        if not matrices_cargadas:
            continue

        for area_id in lista_areas:
            mascara_area = (mask_data == area_id)
            
            if np.any(mascara_area):
                data_fila = {
                    "Mouse": raton, 
                    "Sex": info_sujetos[raton]["Sex"], 
                    "Diet": info_sujetos[raton]["Diet"],
                    "DiffusionTime": tiempo_r, 
                    "Area": area_id
                }
                
                for idx, matriz in enumerate(matrices_cargadas):
                    col_name = f"Mapa_{idx+1}"
                    if matriz is not None:
                        data_fila[col_name] = np.mean(matriz[mascara_area])
                    else:
                        data_fila[col_name] = np.nan
                        
                for idx in range(len(matrices_cargadas), 6):
                    data_fila[f"f'Mapa_{idx+1}'"] = np.nan
                    
                filas_csv.append(data_fila)
                
    print(f"✓ ¡Datos redondeados y extraídos con éxito para el ratón: {raton}!")

# Guardamos el archivo final en tu Escritorio local
if filas_csv:
    df_final = pd.DataFrame(filas_csv)
    desktop_path = r"C:\Users\Estudiantes\Desktop\MD_OUT_NuevasRegiones_Completo.csv"
    df_final.to_csv(desktop_path, index=False)
    print(f"\n🚀 ¡PROCESO COMPLETADO CON ÉXITO!")
    print(f"--> ¡Se han guardado {len(df_final)} filas de datos!")
    print(f"--> Tu CSV está listo en el Escritorio: {desktop_path}")
else:
    print("\n❌ Error inesperado.")

--- INICIANDO EXTRACCIÓN CON CORRECCIÓN DE REDONDEO ---
✓ ¡Datos redondeados y extraídos con éxito para el ratón: R85!
✓ ¡Datos redondeados y extraídos con éxito para el ratón: R86!
✓ ¡Datos redondeados y extraídos con éxito para el ratón: R87!
✓ ¡Datos redondeados y extraídos con éxito para el ratón: R88!
✓ ¡Datos redondeados y extraídos con éxito para el ratón: R98!
✓ ¡Datos redondeados y extraídos con éxito para el ratón: R99!
✓ ¡Datos redondeados y extraídos con éxito para el ratón: R105!
✓ ¡Datos redondeados y extraídos con éxito para el ratón: R107!

🚀 ¡PROCESO COMPLETADO CON ÉXITO!
--> ¡Se han guardado 413 filas de datos!
--> Tu CSV está listo en el Escritorio: C:\Users\Estudiantes\Desktop\MD_OUT_NuevasRegiones_Completo.csv
